# NUS ST3248 - Statistical Learning I
## Cheatsheet-aligned, expanded case study with Palmer Penguins

This notebook rebuilds the earlier Palmer Penguins case study after auditing the uploaded **ST3248 Statistical Learning cheatsheet** (third-party Studocu material, not an official NUS publication).

The aim is not to copy the cheatsheet. Instead, the notebook converts its dense formula map into a **systematic, student-intuitive learning sequence** with executable Python, Bokeh visualisations, interpretation tables, and several special-case experiments.

### Main case study

We use the **Palmer Penguins** dataset for most examples, then introduce a few small synthetic experiments when a concept cannot be demonstrated cleanly from the penguin data alone.

### Learning philosophy

For every method, ask four questions:

1. **What quantity is being optimised?**
2. **What assumptions are being made?**
3. **How do we estimate generalisation?**
4. **How can the method fail?**

The central statistical-learning model is

$$
Y=f(X)+\epsilon,
$$

and our goal is to construct

$$
\hat f(X)\approx f(X)
$$

while controlling out-of-sample error.

# 0. Cheatsheet audit - what was missing from the previous notebook?

The previous notebook already covered regression, polynomial flexibility, K-fold CV, bootstrap, best-subset selection, Ridge/Lasso, classification, ROC, PCA, K-means and hierarchical clustering.

We will cover new areas not previously added.

| Area | Missing / partial concept now added |
|---|---|
| Statistical-learning foundations | parametric vs non-parametric; reducible vs irreducible error; flexibility trade-off |
| Simple regression | manual OLS identities; line passes through $(\bar x,\bar y)$; $R^2=r^2$ special case |
| Regression inference | standard errors; confidence intervals; coefficient $t$ tests; overall $F$ test |
| Prediction uncertainty | confidence interval for mean response vs prediction interval for a new observation |
| Regression diagnostics | non-linearity; heteroscedasticity; correlated errors; outliers; leverage; Cook's distance; VIF |
| Categorical predictors | indicator variables; reference levels; dummy-variable trap |
| Interactions | explicit interaction model and species-specific slopes |
| Model selection | forward, backward and mixed/stepwise ideas; $C_p$, AIC, BIC, adjusted $R^2$ |
| Shrinkage | constrained view; coefficient paths; orthogonal-design Ridge/Lasso special cases |
| High-dimensional regression | $p>n$ interpolation/instability special case |
| Dimension reduction | Principal Components Regression (PCR) and Partial Least Squares (PLS) |
| PCA special case | high-variance PCs need not be predictive of $Y$ |
| Resampling | validation-set approach; LOOCV; linear-model LOOCV leverage shortcut |
| Classification theory | Bayes classifier and Bayes error rate |
| Logistic regression | binary log-odds and odds-ratio interpretation |
| LDA/QDA | class priors, covariance assumptions, decision-boundary comparison |
| KNN | KNN regression and curse-of-dimensionality experiment |
| Classification metrics | sensitivity, specificity, one-vs-rest interpretation |
| Hierarchical clustering | single, complete, average and Ward linkage comparison |

The notebook marks some additions as **special-case teaching extensions**. These extend the cheatsheet's ideas rather than claiming to be verbatim contents of it.

In [1]:
# 1. Setup and reproducibility
import warnings
from itertools import combinations

import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.stats import norm

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, auc, silhouette_score
)
from sklearn.model_selection import (
    KFold, LeaveOneOut, StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler, label_binarize
from sklearn.cluster import KMeans

from bokeh.io import output_notebook, show
from bokeh.layouts import column, gridplot
from bokeh.models import Band, ColumnDataSource, ColorBar, Div, HoverTool, LinearColorMapper
from bokeh.palettes import Category10, Viridis256, RdBu11
from bokeh.plotting import figure
from bokeh.transform import transform

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
output_notebook()
print('Environment ready.')

Loading BokehJS ...

Environment ready.


## Stable Bokeh table helper

Some notebook frontends render Bokeh `DataTable` inconsistently. To avoid the blank-table problem, this notebook renders tables through a Bokeh `Div` containing HTML.

This remains a Bokeh output while being much more portable across Jupyter Notebook, JupyterLab, VS Code and Polyglot notebook environments.

In [2]:
def show_table(df, title=None, width=950, height=None, round_digits=3):
    x = df.copy()
    x.columns = [str(c) for c in x.columns]
    for c in x.select_dtypes(include=np.number).columns:
        x[c] = x[c].round(round_digits)

    table_html = x.to_html(index=False, border=0, classes='st3248-table')
    title_html = f'<h3>{title}</h3>' if title else ''

    html = f'''    <style>
    .st3248-wrap {{font-family:Arial,sans-serif;margin:4px 0 18px 0;}}
    .st3248-table {{border-collapse:collapse;width:100%;font-size:14px;background:white;}}
    .st3248-table th {{background:#f1f5f9;padding:9px 12px;text-align:left;border-bottom:2px solid #94a3b8;}}
    .st3248-table td {{padding:8px 12px;border-bottom:1px solid #e2e8f0;}}
    .st3248-table tr:nth-child(even) {{background:#fafafa;}}
    .st3248-table tr:hover {{background:#f8fafc;}}
    </style>
    <div class='st3248-wrap'>{title_html}{table_html}</div>
    '''
    show(Div(text=html, width=width))


def show_note(title, body, width=950):
    html = f'''    <div style="border-left:5px solid #3b82f6;padding:12px 16px;background:#f8fafc;
                border-radius:6px;font-family:Arial;line-height:1.55;">
      <h3 style="margin:0 0 7px 0;">{title}</h3>{body}
    </div>
    '''
    show(Div(text=html, width=width))


def regression_metrics(y_true, y_pred):
    return {
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }


def classification_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Balanced_Accuracy': balanced_accuracy_score(y_true, y_pred),
        'Macro_F1': f1_score(y_true, y_pred, average='macro'),
    }

# 2. Load the Palmer Penguins dataset

One row represents one penguin.

The notebook first tries the original `palmerpenguins` GitHub source and then a commonly used mirror.

In [3]:
DATA_URLS = [
    'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv',
    'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv',
]

df = None
last_error = None
for url in DATA_URLS:
    try:
        df = pd.read_csv(url, na_values=['NA', ''])
        print('Loaded:', url)
        break
    except Exception as exc:
        last_error = exc

if df is None:
    raise RuntimeError('Could not download Palmer Penguins. Check internet connectivity.') from last_error

print('Shape:', df.shape)
display(df.head())

Loaded: https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv
Shape: (344, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [4]:
data_dictionary = pd.DataFrame({
    'variable': ['species','island','bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g','sex','year'],
    'type': ['categorical','categorical','continuous','continuous','continuous','continuous','categorical','integer'],
    'possible_role': [
        'classification target','contextual predictor','regression/classification predictor',
        'regression/classification predictor','regression/classification predictor',
        'regression target or predictor','categorical predictor','context variable'
    ]
})
show_table(data_dictionary, 'Data dictionary')

quality = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'missing_n': df.isna().sum().values,
    'missing_pct': 100 * df.isna().mean().values,
    'n_unique': df.nunique(dropna=True).values,
})
show_table(quality, 'Data-quality audit')

# 3. Statistical-learning foundations

## 3.1 Prediction vs inference

For prediction, we primarily care about

$$
\hat Y=\hat f(X)
$$

on unseen observations.

For inference, we care about questions such as:

- which predictors matter?
- what is the sign and magnitude of their association?
- how uncertain are the coefficient estimates?
- does the effect of one variable depend on another?

A model can be highly predictive yet difficult to interpret.

## 3.2 Reducible and irreducible error

For

$$
Y=f(X)+\epsilon,
$$

prediction error contains:

- **reducible error** from estimating $f$ imperfectly,
- **irreducible error** from $\epsilon$.

Even the true $f$ cannot remove the irreducible component.

## 3.3 Parametric vs non-parametric

A parametric method assumes a finite-dimensional functional form, for example

$$
f(X)=\beta_0+\beta_1X_1+\cdots+\beta_pX_p.
$$

A non-parametric method such as KNN imposes much less structure but often requires more data and can have higher variance.

# 4. EDA - discovering structure before modelling

In [5]:
species_counts = df['species'].value_counts().rename_axis('species').reset_index(name='count')
p = figure(x_range=species_counts['species'].tolist(), width=700, height=380,
           title='Species frequency', toolbar_location=None)
p.vbar(x=species_counts['species'], top=species_counts['count'], width=0.7)
p.xaxis.axis_label = 'Species'; p.yaxis.axis_label = 'Count'
p.add_tools(HoverTool(tooltips=[('species','@x'),('count','@top')]))
show(p)

In [6]:
plot_df = df.dropna(subset=['flipper_length_mm','body_mass_g','species']).copy()
species_order = sorted(plot_df['species'].unique())
palette = Category10[10]
species_color = {s: palette[i] for i,s in enumerate(species_order)}

p = figure(width=880, height=500, title='Body mass vs flipper length',
           x_axis_label='Flipper length (mm)', y_axis_label='Body mass (g)')
for s in species_order:
    part = plot_df[plot_df['species']==s]
    p.scatter('flipper_length_mm','body_mass_g', source=ColumnDataSource(part),
              size=8, alpha=0.65, color=species_color[s], legend_label=s)
p.legend.location='top_left'
p.add_tools(HoverTool(tooltips=[('species','@species'),('flipper','@flipper_length_mm'),('mass','@body_mass_g')]))
show(p)

In [7]:
num_cols=['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
corr=df[num_cols].corr()
corr_long=corr.stack().rename('correlation').reset_index().rename(columns={'level_0':'x','level_1':'y'})
mapper=LinearColorMapper(palette=RdBu11[::-1],low=-1,high=1)
p=figure(x_range=num_cols,y_range=list(reversed(num_cols)),width=760,height=540,
         title='Correlation matrix',toolbar_location=None,tools='hover',
         tooltips=[('pair','@x × @y'),('correlation','@correlation{0.000}')])
p.rect(x='x',y='y',width=1,height=1,source=ColumnDataSource(corr_long),
       fill_color=transform('correlation',mapper),line_color='white')
p.xaxis.major_label_orientation=0.8
p.add_layout(ColorBar(color_mapper=mapper),'right')
show(p)

In [8]:
num_cols

['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

# 5. Simple linear regression - estimation and special identities

For body mass $Y$ and flipper length $X$:

$$
Y_i=\beta_0+\beta_1X_i+\epsilon_i.
$$

OLS minimises

$$
RSS=\sum_{i=1}^{n}(y_i-\hat y_i)^2.
$$

For simple regression,

$$
\hat\beta_1=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}{\sum_i(x_i-\bar x)^2},
$$

$$
\hat\beta_0=\bar y-\hat\beta_1\bar x.
$$

### Special cases worth remembering

1. The OLS line passes through $(\bar x,\bar y)$.
2. In simple regression with an intercept,

$$
R^2=r_{XY}^2.
$$

In [9]:
reg_df=df[['flipper_length_mm','body_mass_g']].dropna().copy()
x=reg_df['flipper_length_mm'].to_numpy()
y=reg_df['body_mass_g'].to_numpy()

xbar=x.mean(); ybar=y.mean()
b1_manual=((x-xbar)*(y-ybar)).sum()/((x-xbar)**2).sum()
b0_manual=ybar-b1_manual*xbar

lr=LinearRegression().fit(reg_df[['flipper_length_mm']],reg_df['body_mass_g'])
r=np.corrcoef(x,y)[0,1]
r2=lr.score(reg_df[['flipper_length_mm']],reg_df['body_mass_g'])

identity_table=pd.DataFrame({
    'quantity':['manual intercept','sklearn intercept','manual slope','sklearn slope','R2','correlation squared','line at xbar','ybar'],
    'value':[b0_manual,lr.intercept_,b1_manual,lr.coef_[0],r2,r**2,b0_manual+b1_manual*xbar,ybar]
})
show_table(identity_table,'Simple-regression identities')

# 6. Regression inference - standard errors, confidence intervals and tests

Under the classical assumptions, coefficient uncertainty can be quantified.

For simple regression,

$$
SE(\hat\beta_1)
=
\sqrt{\frac{\hat\sigma^2}{\sum_i(x_i-\bar x)^2}}.
$$

The residual standard error is

$$
RSE=\sqrt{\frac{RSS}{n-2}}.
$$

For testing

$$
H_0:\beta_1=0
\qquad\text{vs}\qquad
H_1:\beta_1\neq0,
$$

we use

$$
t=\frac{\hat\beta_1}{SE(\hat\beta_1)}.
$$

In [10]:
X_sm=sm.add_constant(reg_df['flipper_length_mm'])
ols_simple=sm.OLS(reg_df['body_mass_g'],X_sm).fit()
ci=ols_simple.conf_int()
inf_table=pd.DataFrame({
    'term':ols_simple.params.index,
    'estimate':ols_simple.params.values,
    'std_error':ols_simple.bse.values,
    't_stat':ols_simple.tvalues.values,
    'p_value':ols_simple.pvalues.values,
    'CI_2.5%':ci[0].values,
    'CI_97.5%':ci[1].values,
})
show_table(inf_table,'Simple regression: coefficient inference')

rse=np.sqrt(ols_simple.ssr/ols_simple.df_resid)
fit_quality=pd.DataFrame({
    'metric':['RSS','RSE','R2','Adjusted R2','F statistic','F-test p-value','AIC','BIC'],
    'value':[ols_simple.ssr,rse,ols_simple.rsquared,ols_simple.rsquared_adj,ols_simple.fvalue,ols_simple.f_pvalue,ols_simple.aic,ols_simple.bic]
})
show_table(fit_quality,'Quality of fit and overall significance')

In [11]:
ols_simple.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            body_mass_g   R-squared:                       0.759
Model:                            OLS   Adj. R-squared:                  0.758
Method:                 Least Squares   F-statistic:                     1071.
Date:                Sun, 06 Sep 2026   Prob (F-statistic):          4.37e-107
Time:                        11:58:02   Log-Likelihood:                -2528.4
No. Observations:                 342   AIC:                             5061.
Df Residuals:                     340   BIC:                             5069.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const             -5780.8314    305.815    -18.903      0.000   -6382.358   -5179.305
flipper_length_mm    49.6856      1.518     32.722      0.000      46.699      52.672
==============================================================================
Omnibus:                        5.634   Durbin-Watson:                   2.190
Prob(Omnibus):                  0.060   Jarque-Bera (JB):                5.585
Skew:                           0.313   Prob(JB):                       0.0613
Kurtosis:                       3.019   Cond. No.                     2.89e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.89e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

# 7. Confidence interval vs prediction interval

These answer different questions.

### Confidence interval for the mean response

At $x_0$:

$$
E[Y\mid X=x_0].
$$

### Prediction interval for a new individual

For a future observation:

$$
Y_{new}\mid X=x_0.
$$

The prediction interval is wider because it includes both:

- uncertainty in the estimated mean function,
- irreducible individual-level noise.

In [12]:
grid_x=np.linspace(reg_df['flipper_length_mm'].min(),reg_df['flipper_length_mm'].max(),200)
grid_sm=sm.add_constant(pd.DataFrame({'flipper_length_mm':grid_x}),has_constant='add')
pred_frame=ols_simple.get_prediction(grid_sm).summary_frame(alpha=0.05)
plot_ci=pd.DataFrame({
    'x':grid_x,
    'mean':pred_frame['mean'].values,
    'mean_lo':pred_frame['mean_ci_lower'].values,
    'mean_hi':pred_frame['mean_ci_upper'].values,
    'obs_lo':pred_frame['obs_ci_lower'].values,
    'obs_hi':pred_frame['obs_ci_upper'].values,
})

p=figure(width=900,height=500,title='Mean-response CI vs individual prediction interval',
         x_axis_label='Flipper length (mm)',y_axis_label='Body mass (g)')
# Build ColumnDataSource from plot_ci
src = ColumnDataSource(plot_ci)
# Scatter source for raw data
scatter_src = ColumnDataSource(reg_df)

# Scatter glyph
scatter_renderer = p.scatter('flipper_length_mm', 'body_mass_g',
                             source=scatter_src, size=5, alpha=0.25,
                             legend_label='observations')

# Regression line glyph (uses src from plot_ci)
line_renderer = p.line('x', 'mean', source=src, line_width=3,
                       legend_label='fitted mean')

# Bands (also use src from plot_ci)
p.add_layout(Band(base='x', lower='obs_lo', upper='obs_hi',
                  source=src, fill_alpha=0.10, level='underlay',
                  dimension='height'))
p.add_layout(Band(base='x', lower='mean_lo', upper='mean_hi',
                  source=src, fill_alpha=0.25, level='underlay',
                  ))

p.legend.location = 'top_left'

# Hover tool for scatter points
scatter_hover = HoverTool(renderers=[scatter_renderer],
                          tooltips=[('Flipper length', '@flipper_length_mm{0.0}'),
                                    ('Body mass', '@body_mass_g{0.0}')],
                          mode='mouse')

# Hover tool for regression line (values from plot_ci via src)
line_hover = HoverTool(renderers=[line_renderer],
                       tooltips=[('Mean', '@mean{0.0}'),
                                 ('Mean CI low', '@mean_lo{0.0}'),
                                 ('Mean CI high', '@mean_hi{0.0}'),
                                 ('Obs CI low', '@obs_lo{0.0}'),
                                 ('Obs CI high', '@obs_hi{0.0}')],
                       mode='mouse')

# Add both hover tools
p.add_tools(scatter_hover, line_hover)

show(p)

# 8. Multiple regression, categorical variables and the overall F test

A multiple linear model is

$$
Y=\beta_0+\beta_1X_1+\cdots+\beta_pX_p+\epsilon.
$$

An individual coefficient is interpreted while holding the other included predictors fixed.

For testing whether the model has any predictive signal beyond an intercept:

$$
H_0:\beta_1=\cdots=\beta_p=0.
$$

The overall $F$ statistic compares explained and unexplained variation.

Categorical predictors are represented through indicator variables. One level is omitted as the reference level to avoid the **dummy-variable trap** (perfect collinearity with the intercept).

In [13]:
reg_model_df=df[['body_mass_g','bill_length_mm','bill_depth_mm','flipper_length_mm','sex','island','year','species']].dropna().copy()
formula='body_mass_g ~ bill_length_mm + bill_depth_mm + flipper_length_mm + C(sex) + C(island) + year'
ols_multi=smf.ols(formula,data=reg_model_df).fit()
ci=ols_multi.conf_int()
coef_table=pd.DataFrame({
    'term':ols_multi.params.index,
    'estimate':ols_multi.params.values,
    'std_error':ols_multi.bse.values,
    't_stat':ols_multi.tvalues.values,
    'p_value':ols_multi.pvalues.values,
    'CI_low':ci[0].values,
    'CI_high':ci[1].values,
})
show_table(coef_table,'Multiple-regression coefficient table',height=380)

summary=pd.DataFrame({
    'metric':['R2','Adjusted R2','RSS','RSE','F statistic','F-test p-value'],
    'value':[ols_multi.rsquared,ols_multi.rsquared_adj,ols_multi.ssr,
             np.sqrt(ols_multi.ssr/ols_multi.df_resid),ols_multi.fvalue,ols_multi.f_pvalue]
})
show_table(summary,'Multiple-regression model summary')

# 9. Interaction terms - when one effect depends on another variable

The additive model assumes the effect of $X_1$ does not depend on $X_2$.

An interaction model allows

$$
Y=\beta_0+\beta_1X+\beta_2S+\beta_3XS+\epsilon.
$$

Then the slope with respect to $X$ depends on $S$.

We test whether the body-mass/flipper-length relationship differs by species.

In [14]:
interaction_model=smf.ols('body_mass_g ~ flipper_length_mm * C(species)',data=reg_model_df).fit()
coef_inter=pd.DataFrame({
    'term':interaction_model.params.index,
    'estimate':interaction_model.params.values,
    'p_value':interaction_model.pvalues.values,
})
show_table(coef_inter,'Flipper length × species interaction coefficients')

base_slope=interaction_model.params['flipper_length_mm']
slopes={'Adelie/reference':base_slope}
for sp in ['Chinstrap','Gentoo']:
    term=f'flipper_length_mm:C(species)[T.{sp}]'
    slopes[sp]=base_slope + interaction_model.params.get(term,0.0)
show_table(pd.DataFrame({'species':slopes.keys(),'estimated_flipper_slope_g_per_mm':slopes.values()}),
           'Species-specific fitted slopes')

# 10. Regression problems and diagnostics

The cheatsheet highlights common problems with linear regression:

1. non-linearity,
2. correlation of error terms,
3. non-constant error variance,
4. outliers,
5. high-leverage observations,
6. collinearity.

We can inspect several of these directly.

## 10.1 Residuals

A residual is

$$
e_i=y_i-\hat y_i.
$$

Curvature can suggest nonlinearity. A funnel shape can suggest heteroscedasticity.

In [15]:
resid_df=pd.DataFrame({'fitted':ols_multi.fittedvalues,'residual':ols_multi.resid})
p=figure(width=850,height=420,title='Residuals vs fitted values',
         x_axis_label='Fitted body mass',y_axis_label='Residual')
p.scatter('fitted','residual',source=ColumnDataSource(resid_df),size=7,alpha=0.55)
p.line([resid_df['fitted'].min(),resid_df['fitted'].max()],[0,0],line_dash='dashed',line_width=2)
show(p)

## 10.2 Outliers, leverage and Cook's distance

An observation can be unusual in different ways.

- **Outlier:** unusual response given the fitted model.
- **High leverage:** unusual predictor configuration.
- **Influential point:** deleting it materially changes the fitted model.

Leverage is given by the diagonal of the hat matrix:

$$
h_{ii}=x_i^T(X^TX)^{-1}x_i.
$$

Cook's distance combines residual size and leverage.

In [16]:
infl=ols_multi.get_influence()
inf_frame=infl.summary_frame()
diag=pd.DataFrame({
    'leverage':inf_frame['hat_diag'],
    'studentized_residual':inf_frame['student_resid'],
    'cooks_d':inf_frame['cooks_d'],
})

p=figure(width=850,height=450,title='Leverage vs studentized residual',
         x_axis_label='Leverage h_ii',y_axis_label='Studentized residual')
p.scatter('leverage','studentized_residual',source=ColumnDataSource(diag),
          size=7,alpha=0.55)
p.line([diag['leverage'].min(),diag['leverage'].max()],[2,2],line_dash='dashed')
p.line([diag['leverage'].min(),diag['leverage'].max()],[-2,-2],line_dash='dashed')
show(p)

show_table(diag.nlargest(10,'cooks_d').reset_index(drop=True),"Ten most influential observations by Cook's distance")

## 10.3 Multicollinearity and VIF

If predictor $X_j$ can be well predicted by the remaining predictors, its coefficient can become unstable.

A common diagnostic is

$$
VIF_j=\frac{1}{1-R_j^2},
$$

where $R_j^2$ is obtained by regressing $X_j$ on the other predictors.

Large VIF does not necessarily destroy prediction, but it can inflate coefficient uncertainty and make inference unstable.

In [17]:
X_vif=pd.get_dummies(reg_model_df[['bill_length_mm','bill_depth_mm','flipper_length_mm','year','sex','island']],drop_first=True,dtype=float)
X_vif_const=sm.add_constant(X_vif)
vif_rows=[]
for j,name in enumerate(X_vif_const.columns):
    if name=='const':
        continue
    vif_rows.append({'feature':name,'VIF':variance_inflation_factor(X_vif_const.values,j)})
show_table(pd.DataFrame(vif_rows).sort_values('VIF',ascending=False),'Variance Inflation Factors')

## 10.4 Correlated errors - special-case teaching extension

The penguin observations are essentially cross-sectional, so serial dependence is not naturally demonstrated here.

In time-ordered data, however,

$$
Cov(\epsilon_t,\epsilon_{t-1})\neq0
$$

can make ordinary regression standard errors misleading.

The small synthetic example below compares independent errors with AR(1)-style correlated errors.

In [18]:
n=180
x_syn=np.linspace(0,10,n)
ind_err=rng.normal(0,1,n)
cor_err=np.zeros(n)
for t in range(1,n):
    cor_err[t]=0.85*cor_err[t-1]+rng.normal(0,0.55)

y_ind=2+1.5*x_syn+ind_err
y_cor=2+1.5*x_syn+cor_err

m_ind=sm.OLS(y_ind,sm.add_constant(x_syn)).fit()
m_cor=sm.OLS(y_cor,sm.add_constant(x_syn)).fit()
show_table(pd.DataFrame({
    'scenario':['independent errors','correlated errors'],
    'slope_estimate':[m_ind.params[1],m_cor.params[1]],
    'naive_slope_SE':[m_ind.bse[1],m_cor.bse[1]],
    'Durbin_Watson':[sm.stats.stattools.durbin_watson(m_ind.resid),sm.stats.stattools.durbin_watson(m_cor.resid)]
}),'Why correlated residuals matter')

# 11. Flexibility and the bias-variance trade-off

Increasing flexibility usually lowers training error.

But test error can behave non-monotonically because

$$
E[(Y_0-\hat f(X_0))^2]
=
Var(\hat f(X_0))
+
Bias(\hat f(X_0))^2
+
Var(\epsilon).
$$

We use polynomial degree as a direct complexity control.

In [19]:
poly_df=df[['flipper_length_mm','body_mass_g']].dropna()
X_poly=poly_df[['flipper_length_mm']]
y_poly=poly_df['body_mass_g']
cv10=KFold(n_splits=10,shuffle=True,random_state=RANDOM_STATE)
poly_rows=[]
for degree in range(1,11):
    model=Pipeline([
        ('poly',PolynomialFeatures(degree=degree,include_bias=False)),
        ('scale',StandardScaler()),
        ('model',LinearRegression())
    ])
    rmse=np.sqrt(-cross_val_score(model,X_poly,y_poly,cv=cv10,scoring='neg_mean_squared_error'))
    model.fit(X_poly,y_poly)
    train_rmse=mean_squared_error(y_poly,model.predict(X_poly))**0.5
    poly_rows.append({'degree':degree,'training_RMSE':train_rmse,'CV_RMSE':rmse.mean(),'CV_sd':rmse.std(ddof=1)})
poly_results=pd.DataFrame(poly_rows)
show_table(poly_results,'Training vs CV error as flexibility increases')

p=figure(width=850,height=430,title='Training error keeps falling; CV error need not',
         x_axis_label='Polynomial degree',y_axis_label='RMSE')
p.line(poly_results['degree'],poly_results['training_RMSE'],line_width=3,legend_label='training RMSE')
p.line(poly_results['degree'],poly_results['CV_RMSE'],line_width=3,legend_label='10-fold CV RMSE')
p.scatter(poly_results['degree'],poly_results['CV_RMSE'],size=8)
p.legend.location='top_right'
show(p)

# 12. Resampling: validation set, LOOCV and K-fold CV

## Validation-set approach

Split data once into training and validation subsets.

Pros: simple and fast.

Cons: result can depend strongly on one random split and uses fewer observations for fitting.

## Leave-one-out cross-validation

LOOCV is the special case $K=n$.

Each model is trained on $n-1$ observations and tested on the one omitted observation.

## K-fold CV

Usually $K=5$ or $10$ gives a useful computational/bias-variance compromise.

In [20]:
resample_rows=[]
for seed in range(30):
    Xtr,Xva,ytr,yva=train_test_split(X_poly,y_poly,test_size=0.25,random_state=seed)
    m=LinearRegression().fit(Xtr,ytr)
    resample_rows.append({'method':'validation split','estimate':mean_squared_error(yva,m.predict(Xva))**0.5})

loo=LeaveOneOut()
loo_rmse=np.sqrt(-cross_val_score(LinearRegression(),X_poly,y_poly,cv=loo,scoring='neg_mean_squared_error').mean())
resample_rows.append({'method':'LOOCV','estimate':loo_rmse})
for k in [5,10]:
    cv=KFold(n_splits=k,shuffle=True,random_state=RANDOM_STATE)
    rmse=np.sqrt(-cross_val_score(LinearRegression(),X_poly,y_poly,cv=cv,scoring='neg_mean_squared_error'))
    resample_rows.append({'method':f'{k}-fold CV','estimate':rmse.mean()})

resample_df=pd.DataFrame(resample_rows)
show_table(resample_df.groupby('method')['estimate'].agg(['mean','std','min','max']).reset_index(),
           'Resampling methods: estimated RMSE')

## 12.1 LOOCV shortcut for ordinary linear regression - special case

For OLS, the leave-one-out residual can be obtained without refitting $n$ models:

$$
e_{(i)}=\frac{e_i}{1-h_{ii}}.
$$

Therefore

$$
CV_{LOO}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left(\frac{e_i}{1-h_{ii}}\right)^2.
$$

This elegant identity connects **cross-validation** to **leverage**.

In [21]:
X_hat=sm.add_constant(reg_df['flipper_length_mm'])
ols_hat=sm.OLS(reg_df['body_mass_g'],X_hat).fit()
hat_diag=ols_hat.get_influence().hat_matrix_diag
loo_mse_hat=np.mean((ols_hat.resid/(1-hat_diag))**2)
loo_rmse_hat=np.sqrt(loo_mse_hat)
loo_rmse_refit=np.sqrt(-cross_val_score(LinearRegression(),reg_df[['flipper_length_mm']],reg_df['body_mass_g'],
                                        cv=LeaveOneOut(),scoring='neg_mean_squared_error').mean())
show_table(pd.DataFrame({
    'LOOCV_computation':['hat-matrix shortcut','explicit refitting'],
    'RMSE':[loo_rmse_hat,loo_rmse_refit]
}),'OLS LOOCV special-case identity')

# 13. Bootstrap - computational uncertainty estimation

Bootstrap repeatedly samples observations **with replacement** from the observed data.

For an estimator $\hat\theta$, bootstrap replicates

$$
\hat\theta^{*(1)},\ldots,\hat\theta^{*(B)}
$$

approximate its sampling distribution.

We compare the bootstrap standard error of the regression slope with the analytical OLS standard error.

In [22]:
boot_df=reg_df.reset_index(drop=True)
B=1000
slopes=np.empty(B)
for b in range(B):
    idx=rng.integers(0,len(boot_df),len(boot_df))
    sample=boot_df.iloc[idx]
    m=LinearRegression().fit(sample[['flipper_length_mm']],sample['body_mass_g'])
    slopes[b]=m.coef_[0]

boot_se=slopes.std(ddof=1)
analytic_se=ols_simple.bse['flipper_length_mm']
ci_low,ci_high=np.percentile(slopes,[2.5,97.5])
show_table(pd.DataFrame({
    'method':['analytic OLS SE','bootstrap SE','bootstrap percentile CI lower','bootstrap percentile CI upper'],
    'value':[analytic_se,boot_se,ci_low,ci_high]
}),'Analytical vs bootstrap uncertainty')

hist,edges=np.histogram(slopes,bins=35)
hist_df=pd.DataFrame({'left':edges[:-1],'right':edges[1:],'count':hist})
p=figure(width=850,height=420,title='Bootstrap distribution of the flipper-length slope',
         x_axis_label='Slope (grams per mm)',y_axis_label='Frequency')
p.quad(top='count',bottom=0,left='left',right='right',source=ColumnDataSource(hist_df),alpha=0.65)
p.line([ci_low,ci_low],[0,hist.max()],line_dash='dashed',line_width=2)
p.line([ci_high,ci_high],[0,hist.max()],line_dash='dashed',line_width=2)
show(p)

# 14. Model selection: best subset, $C_p$, AIC, BIC and adjusted $R^2$

Training RSS always decreases when predictors are added. Therefore RSS alone is not a valid way to compare models of different sizes.

Common criteria penalise unnecessary complexity.

### Mallows $C_p$

$$
C_p=\frac{RSS_p}{\hat\sigma^2}-n+2d,
$$

where $d$ is the number of fitted coefficients including the intercept.

### AIC and BIC

Both balance fit and complexity. BIC penalises model size more strongly as $n$ grows.

### Adjusted $R^2$

Unlike ordinary $R^2$, adjusted $R^2$ can decrease when an unhelpful variable is added.

In [23]:
sel_df=df[['body_mass_g','bill_length_mm','bill_depth_mm','flipper_length_mm','year','sex']].dropna().copy()
X_sel=pd.get_dummies(sel_df.drop(columns='body_mass_g'),drop_first=True,dtype=float)
y_sel=sel_df['body_mass_g'].astype(float)

X_full=sm.add_constant(X_sel)
full_fit=sm.OLS(y_sel,X_full).fit()
sigma2_full=full_fit.ssr/full_fit.df_resid
n=len(y_sel)
rows=[]
for r in range(1,X_sel.shape[1]+1):
    for cols in combinations(X_sel.columns,r):
        Xc=sm.add_constant(X_sel[list(cols)])
        fit=sm.OLS(y_sel,Xc).fit()
        d=Xc.shape[1]
        cp=fit.ssr/sigma2_full - n + 2*d
        rows.append({
            'n_features':r,
            'features':', '.join(cols),
            'RSS':fit.ssr,
            'Cp':cp,
            'AIC':fit.aic,
            'BIC':fit.bic,
            'Adjusted_R2':fit.rsquared_adj,
        })
subset_criteria=pd.DataFrame(rows)
best_size=[]
for r,g in subset_criteria.groupby('n_features'):
    best_size.append({
        'n_features':r,
        'best_AIC':g['AIC'].min(),
        'best_BIC':g['BIC'].min(),
        'best_Cp':g['Cp'].min(),
        'best_Adjusted_R2':g['Adjusted_R2'].max(),
        'AIC_model':g.loc[g['AIC'].idxmin(),'features'],
    })
show_table(pd.DataFrame(best_size),'Best model at each size under common criteria')

# 15. Forward, backward and mixed/stepwise selection

### Forward selection

Start from the intercept-only model and repeatedly add the predictor giving the largest improvement.

### Backward selection

Start from the full model and repeatedly remove a predictor.

Backward selection requires that the full model be estimable, so it is problematic when $p\ge n$.

### Mixed selection

Alternate forward additions and backward deletions.

The implementation below uses **AIC** as the comparison criterion so the process is reproducible and explicit.

In [24]:
def fit_aic(X,y,features):
    Xd=sm.add_constant(X[features],has_constant='add') if features else np.ones((len(y),1))
    return sm.OLS(y,Xd).fit().aic


def forward_aic(X,y):
    remaining=list(X.columns); selected=[]; history=[]
    current=fit_aic(X,y,selected)
    while remaining:
        candidates=[]
        for f in remaining:
            aic=fit_aic(X,y,selected+[f])
            candidates.append((aic,f))
        best_aic,best_f=min(candidates)
        if best_aic < current - 1e-9:
            selected.append(best_f); remaining.remove(best_f); current=best_aic
            history.append({'step':len(history)+1,'action':f'add {best_f}','AIC':current,'features':', '.join(selected)})
        else:
            break
    return selected,pd.DataFrame(history)


def backward_aic(X,y):
    selected=list(X.columns); history=[]; current=fit_aic(X,y,selected)
    while len(selected)>1:
        candidates=[]
        for f in selected:
            trial=[x for x in selected if x!=f]
            candidates.append((fit_aic(X,y,trial),f,trial))
        best_aic,removed,trial=min(candidates,key=lambda t:t[0])
        if best_aic < current - 1e-9:
            selected=trial; current=best_aic
            history.append({'step':len(history)+1,'action':f'remove {removed}','AIC':current,'features':', '.join(selected)})
        else:
            break
    return selected,pd.DataFrame(history)

forward_features,forward_history=forward_aic(X_sel,y_sel)
backward_features,backward_history=backward_aic(X_sel,y_sel)
show_table(forward_history,'Forward AIC selection path')
show_table(backward_history,'Backward AIC selection path')
show_table(pd.DataFrame({'method':['forward','backward'],'selected_features':[', '.join(forward_features),', '.join(backward_features)]}),
           'Final stepwise selections')

# 16. Ridge and Lasso shrinkage

Ridge solves

$$
\min_\beta\left[RSS+\lambda\sum_{j=1}^{p}\beta_j^2\right].
$$

Lasso solves

$$
\min_\beta\left[RSS+\lambda\sum_{j=1}^{p}|\beta_j|\right].
$$

Equivalent constrained forms are

$$
\sum_j\beta_j^2\le s
$$

for Ridge and

$$
\sum_j|\beta_j|\le s
$$

for Lasso.

Because penalties act on coefficient magnitude, predictors should normally be standardised first.

In [25]:
alphas=np.logspace(-3,4,45)
cv=KFold(n_splits=10,shuffle=True,random_state=RANDOM_STATE)
ridge_rows=[]; lasso_rows=[]
for a in alphas:
    ridge=Pipeline([('scale',StandardScaler()),('model',Ridge(alpha=a))])
    lasso=Pipeline([('scale',StandardScaler()),('model',Lasso(alpha=a,max_iter=20000))])
    rr=np.sqrt(-cross_val_score(ridge,X_sel,y_sel,cv=cv,scoring='neg_mean_squared_error')).mean()
    lr_=np.sqrt(-cross_val_score(lasso,X_sel,y_sel,cv=cv,scoring='neg_mean_squared_error')).mean()
    ridge_rows.append({'alpha':a,'CV_RMSE':rr}); lasso_rows.append({'alpha':a,'CV_RMSE':lr_})
ridge_cv=pd.DataFrame(ridge_rows); lasso_cv=pd.DataFrame(lasso_rows)
p=figure(x_axis_type='log',width=860,height=430,title='Regularisation strength vs CV RMSE',
         x_axis_label='alpha / lambda',y_axis_label='CV RMSE')
p.line(ridge_cv['alpha'],ridge_cv['CV_RMSE'],line_width=3,legend_label='Ridge')
p.line(lasso_cv['alpha'],lasso_cv['CV_RMSE'],line_width=3,legend_label='Lasso')
p.legend.location='top_left'; show(p)

In [26]:
scaler_paths=StandardScaler()
Xs=scaler_paths.fit_transform(X_sel)
ridge_paths={c:[] for c in X_sel.columns}; lasso_paths={c:[] for c in X_sel.columns}
for a in alphas:
    r=Ridge(alpha=a).fit(Xs,y_sel)
    l=Lasso(alpha=a,max_iter=20000).fit(Xs,y_sel)
    for j,c in enumerate(X_sel.columns):
        ridge_paths[c].append(r.coef_[j]); lasso_paths[c].append(l.coef_[j])

p1=figure(x_axis_type='log',width=850,height=420,title='Ridge coefficient paths',x_axis_label='alpha',y_axis_label='standardized coefficient')
p2=figure(x_axis_type='log',width=850,height=420,title='Lasso coefficient paths',x_axis_label='alpha',y_axis_label='standardized coefficient')
for c in X_sel.columns:
    p1.line(alphas,ridge_paths[c],line_width=2,legend_label=c)
    p2.line(alphas,lasso_paths[c],line_width=2,legend_label=c)
p1.legend.click_policy='hide'; p2.legend.click_policy='hide'
show(column(p1,p2))

## 16.1 Orthogonal-design special cases

If the design columns are orthonormal,

$$
X^TX=I,
$$

Ridge has the particularly transparent solution

$$
\hat\beta_j^{ridge}
=
\frac{\hat\beta_j^{OLS}}{1+\lambda}.
$$

For the objective

$$
\frac12\|y-X\beta\|_2^2+\lambda\|\beta\|_1,
$$

Lasso reduces to **soft thresholding**:

$$
\hat\beta_j^{lasso}
=
\operatorname{sign}(z_j)(|z_j|-\lambda)_+,
$$

where $z_j=\hat\beta_j^{OLS}$ under orthonormality.

In [27]:
A=rng.normal(size=(80,3))
Q,_=np.linalg.qr(A)
beta_true=np.array([3.0,1.2,0.25])
y_orth=Q@beta_true+rng.normal(0,0.08,80)
beta_ols=Q.T@y_orth
lam=0.6
beta_ridge_closed=beta_ols/(1+lam)
beta_lasso_soft=np.sign(beta_ols)*np.maximum(np.abs(beta_ols)-lam,0)
show_table(pd.DataFrame({
    'coefficient':['beta1','beta2','beta3'],
    'OLS':beta_ols,
    'Ridge_closed_form':beta_ridge_closed,
    'Lasso_soft_threshold':beta_lasso_soft,
}),'Orthogonal-design shrinkage special case')

# 17. High-dimensional special case: $p>n$

When the number of predictors exceeds the number of training observations, ordinary least squares can interpolate the training data and the coefficient vector is not uniquely identified without an additional convention.

This is exactly where shrinkage becomes especially valuable.

In [28]:
n_train=45; n_test=600; p_dim=90
beta=np.zeros(p_dim); beta[:8]=rng.normal(0,2,8)
Xtr=rng.normal(size=(n_train,p_dim)); Xte=rng.normal(size=(n_test,p_dim))
ytr=Xtr@beta+rng.normal(0,1,n_train); yte=Xte@beta+rng.normal(0,1,n_test)
models_hd={
    'OLS minimum-norm':LinearRegression(),
    'Ridge alpha=10':Ridge(alpha=10),
    'Lasso alpha=0.1':Lasso(alpha=0.1,max_iter=30000),
}
rows=[]
for name,m in models_hd.items():
    m.fit(Xtr,ytr)
    rows.append({
        'model':name,
        'train_RMSE':mean_squared_error(ytr,m.predict(Xtr))**0.5,
        'test_RMSE':mean_squared_error(yte,m.predict(Xte))**0.5,
        'nonzero_coefficients':np.sum(np.abs(m.coef_)>1e-8),
    })
show_table(pd.DataFrame(rows),'p > n: interpolation vs regularisation')

# 18. Dimension reduction for regression: PCR and PLS

## Principal Components Regression (PCR)

1. standardise predictors,
2. compute principal components of $X$,
3. regress $Y$ on the first $M$ components,
4. select $M$ using cross-validation.

PCR chooses directions that explain predictor variance, **not** directions that are necessarily predictive of $Y$.

## Partial Least Squares (PLS)

PLS is supervised: it constructs latent directions using information from both $X$ and $Y$.

In [29]:
pcr_df=df[['body_mass_g','bill_length_mm','bill_depth_mm','flipper_length_mm','year']].dropna().copy()
X_dr=pcr_df.drop(columns='body_mass_g'); y_dr=pcr_df['body_mass_g']
cv=KFold(n_splits=10,shuffle=True,random_state=RANDOM_STATE)
rows=[]
max_comp=X_dr.shape[1]
for m in range(1,max_comp+1):
    pcr=Pipeline([('scale',StandardScaler()),('pca',PCA(n_components=m)),('lr',LinearRegression())])
    pls=PLSRegression(n_components=m,scale=True)
    pcr_rmse=np.sqrt(-cross_val_score(pcr,X_dr,y_dr,cv=cv,scoring='neg_mean_squared_error')).mean()
    pls_rmse=np.sqrt(-cross_val_score(pls,X_dr,y_dr,cv=cv,scoring='neg_mean_squared_error')).mean()
    rows.append({'components':m,'PCR_CV_RMSE':pcr_rmse,'PLS_CV_RMSE':pls_rmse})
dr_results=pd.DataFrame(rows)
show_table(dr_results,'PCR vs PLS by number of latent components')
p=figure(width=850,height=420,title='PCR vs PLS cross-validated prediction error',x_axis_label='Number of components',y_axis_label='CV RMSE')
p.line(dr_results['components'],dr_results['PCR_CV_RMSE'],line_width=3,legend_label='PCR')
p.line(dr_results['components'],dr_results['PLS_CV_RMSE'],line_width=3,legend_label='PLS')
p.scatter(dr_results['components'],dr_results['PCR_CV_RMSE'],size=8)
p.scatter(dr_results['components'],dr_results['PLS_CV_RMSE'],size=8)
p.legend.location='top_right'; show(p)

## 18.1 PCA can discard a predictive low-variance direction - special case

A major conceptual trap is to assume:

> the direction with the most predictor variance must be the direction most useful for prediction.

That is false.

We create two highly correlated variables. Their common direction has high variance, while their small difference has low variance. The response depends on that low-variance contrast.

PCR with only the first PC can therefore perform poorly, while PLS can use the response when constructing its first latent direction.

In [30]:
n=500
z=rng.normal(0,4,n)
e=rng.normal(0,0.35,n)
X_counter=pd.DataFrame({'X1':z+e,'X2':z-e})
y_counter=8*e+rng.normal(0,0.8,n)
cv=KFold(n_splits=10,shuffle=True,random_state=RANDOM_STATE)
models={
    'PCR 1 PC':Pipeline([('scale',StandardScaler()),('pca',PCA(n_components=1)),('lr',LinearRegression())]),
    'PCR 2 PCs':Pipeline([('scale',StandardScaler()),('pca',PCA(n_components=2)),('lr',LinearRegression())]),
    'PLS 1 component':PLSRegression(n_components=1,scale=True),
    'OLS':Pipeline([('scale',StandardScaler()),('lr',LinearRegression())]),
}
rows=[]
for name,m in models.items():
    rmse=np.sqrt(-cross_val_score(m,X_counter,y_counter,cv=cv,scoring='neg_mean_squared_error')).mean()
    rows.append({'model':name,'CV_RMSE':rmse})
show_table(pd.DataFrame(rows).sort_values('CV_RMSE'),'Counterexample: high-variance PCs need not be predictive')

# 19. Classification foundations: Bayes classifier and Bayes error

The Bayes classifier assigns $x$ to the class with the largest posterior probability:

$$
\hat y(x)=\arg\max_k P(Y=k\mid X=x).
$$

Even the Bayes classifier can make mistakes when class distributions overlap. Its error is the **Bayes error rate**, the irreducible classification error.

In [31]:
mu0,mu1=-1.2,1.2
sigma=1.0
prior0=prior1=0.5
xg=np.linspace(-5,5,500)
d0=prior0*norm.pdf(xg,mu0,sigma)
d1=prior1*norm.pdf(xg,mu1,sigma)
boundary=(mu0+mu1)/2
bayes_error=0.5*(1-norm.cdf(boundary,mu0,sigma))+0.5*norm.cdf(boundary,mu1,sigma)

p=figure(width=850,height=420,title='Bayes decision boundary for two overlapping Gaussian classes',x_axis_label='x',y_axis_label='prior × density')
p.line(xg,d0,line_width=3,legend_label='Class 0')
p.line(xg,d1,line_width=3,legend_label='Class 1')
p.line([boundary,boundary],[0,max(d0.max(),d1.max())],line_dash='dashed',line_width=2,legend_label='Bayes boundary')
p.legend.location='top_left'; show(p)
show_note('Bayes error',f'Because the two class distributions overlap, even the optimal Bayes rule has error approximately <b>{bayes_error:.3f}</b>.')

# 20. Binary logistic regression - log-odds and odds ratios

For a binary response,

$$
p(X)=P(Y=1\mid X),
$$

and logistic regression models

$$
\log\frac{p(X)}{1-p(X)}
=
\beta_0+\beta_1X_1+\cdots+\beta_pX_p.
$$

A one-unit increase in $X_j$ multiplies the odds by

$$
e^{\beta_j}
$$

when other predictors are held fixed.

In [32]:
bin_df=df[['species','bill_length_mm','bill_depth_mm','flipper_length_mm']].dropna().copy()
bin_df['is_gentoo']=(bin_df['species']=='Gentoo').astype(int)
logit_fit=smf.logit('is_gentoo ~ bill_length_mm + bill_depth_mm',data=bin_df).fit(disp=False)
logit_table=pd.DataFrame({
    'term':logit_fit.params.index,
    'log_odds_coefficient':logit_fit.params.values,
    'odds_ratio_exp_beta':np.exp(logit_fit.params.values),
    'p_value':logit_fit.pvalues.values,
})
show_table(logit_table,'Binary logistic regression: coefficient and odds-ratio views')

# 21. Multiclass classification: Logistic, LDA, QDA and KNN

### LDA

Assumes

$$
X\mid Y=k\sim N(\mu_k,\Sigma)
$$

with a common covariance matrix. This creates linear decision boundaries.

### QDA

Assumes

$$
X\mid Y=k\sim N(\mu_k,\Sigma_k),
$$

allowing class-specific covariance matrices and quadratic boundaries.

QDA is more flexible but estimates many more covariance parameters and can therefore have higher variance.

### KNN

For test point $x_0$:

$$
P(Y=j\mid X=x_0)
\approx
\frac1K\sum_{i\in N_0}I(y_i=j).
$$

In [33]:
class_features=['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
class_df=df[class_features+['species']].dropna().copy()
Xc=class_df[class_features]; yc=class_df['species']
Xc_train,Xc_test,yc_train,yc_test=train_test_split(Xc,yc,test_size=0.25,stratify=yc,random_state=RANDOM_STATE)
models={
    'Logistic Regression':Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=5000))]),
    'LDA':Pipeline([('scale',StandardScaler()),('model',LinearDiscriminantAnalysis())]),
    'QDA':Pipeline([('scale',StandardScaler()),('model',QuadraticDiscriminantAnalysis(reg_param=0.01))]),
    'KNN K=7':Pipeline([('scale',StandardScaler()),('model',KNeighborsClassifier(n_neighbors=7))]),
}
skf=StratifiedKFold(n_splits=10,shuffle=True,random_state=RANDOM_STATE)
rows=[]
for name,m in models.items():
    acc=cross_val_score(m,Xc,yc,cv=skf,scoring='accuracy')
    bal=cross_val_score(m,Xc,yc,cv=skf,scoring='balanced_accuracy')
    f1=cross_val_score(m,Xc,yc,cv=skf,scoring='f1_macro')
    rows.append({'model':name,'accuracy':acc.mean(),'balanced_accuracy':bal.mean(),'macro_F1':f1.mean(),'macro_F1_sd':f1.std(ddof=1)})
benchmark=pd.DataFrame(rows).sort_values('macro_F1',ascending=False).reset_index(drop=True)
show_table(benchmark,'10-fold CV classifier comparison')

# 22. Decision boundaries: Logistic vs LDA vs QDA vs KNN

To make decision-boundary geometry visible, we temporarily use only two predictors:

- bill length,
- bill depth.

The goal here is not to maximise accuracy. It is to visualise how modelling assumptions change the boundary shape.

In [34]:
X2=class_df[['bill_length_mm','bill_depth_mm']]
y2=class_df['species']
models2={
    'Logistic':Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=5000))]),
    'LDA':Pipeline([('scale',StandardScaler()),('model',LinearDiscriminantAnalysis())]),
    'QDA':Pipeline([('scale',StandardScaler()),('model',QuadraticDiscriminantAnalysis(reg_param=0.01))]),
    'KNN':Pipeline([('scale',StandardScaler()),('model',KNeighborsClassifier(n_neighbors=9))]),
}
for m in models2.values(): m.fit(X2,y2)

x1=np.linspace(X2.iloc[:,0].min()-1,X2.iloc[:,0].max()+1,130)
x2=np.linspace(X2.iloc[:,1].min()-1,X2.iloc[:,1].max()+1,130)
xx,yy=np.meshgrid(x1,x2)
grid=pd.DataFrame({'bill_length_mm':xx.ravel(),'bill_depth_mm':yy.ravel()})
colors={s:species_color[s] for s in sorted(y2.unique())}
figs=[]
for name,m in models2.items():
    gp=m.predict(grid)
    grid_plot=grid.copy(); grid_plot['pred']=gp; grid_plot['color']=[colors[v] for v in gp]
    p=figure(width=430,height=380,title=name,x_axis_label='Bill length',y_axis_label='Bill depth')
    p.scatter('bill_length_mm','bill_depth_mm',source=ColumnDataSource(grid_plot),size=3,alpha=0.08,color='color')
    for s in sorted(y2.unique()):
        part=class_df[class_df['species']==s]
        p.scatter('bill_length_mm','bill_depth_mm',source=ColumnDataSource(part),size=6,alpha=0.75,color=colors[s],legend_label=s)
    p.legend.location='top_left'; figs.append(p)
show(gridplot([figs[:2],figs[2:]]))

# 23. Classification metrics: sensitivity, specificity and ROC/AUC

For a one-vs-rest class:

$$
Sensitivity=\frac{TP}{TP+FN},
$$

$$
Specificity=\frac{TN}{TN+FP}.
$$

ROC varies the decision threshold and plots TPR against FPR.

A random ranking has expected AUC near $0.5$; larger AUC indicates better ranking performance.

In [35]:
best_name=benchmark.loc[0,'model']; best=clone(models[best_name]).fit(Xc_train,yc_train)
pred=best.predict(Xc_test)
labels=sorted(yc.unique())
cm=confusion_matrix(yc_test,pred,labels=labels)
metric_rows=[]
for i,cls in enumerate(labels):
    TP=cm[i,i]; FN=cm[i,:].sum()-TP; FP=cm[:,i].sum()-TP; TN=cm.sum()-TP-FN-FP
    metric_rows.append({'class':cls,'sensitivity_recall':TP/(TP+FN),'specificity':TN/(TN+FP)})
show_table(pd.DataFrame(metric_rows),f'One-vs-rest sensitivity and specificity - {best_name}')

In [36]:
roc_model=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=5000))]).fit(Xc_train,yc_train)
proba=roc_model.predict_proba(Xc_test)
classes=roc_model.named_steps['model'].classes_
y_bin=label_binarize(yc_test,classes=classes)
p=figure(width=760,height=500,title='One-vs-rest ROC curves',x_axis_label='False positive rate',y_axis_label='True positive rate')
for i,cls in enumerate(classes):
    fpr,tpr,_=roc_curve(y_bin[:,i],proba[:,i]); a=auc(fpr,tpr)
    p.line(fpr,tpr,line_width=3,legend_label=f'{cls} AUC={a:.3f}')
p.line([0,1],[0,1],line_dash='dashed',line_width=2)
p.legend.location='bottom_right'; show(p)

# 24. KNN regression

KNN is not limited to classification.

For regression, predict by averaging the responses of the $K$ nearest training observations:

$$
\hat f(x_0)=\frac1K\sum_{i\in N_0}y_i.
$$

Small $K$ is flexible and high-variance. Large $K$ produces a smoother, higher-bias fit.

In [37]:
knn_reg_df=df[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']].dropna().copy()
Xkr=knn_reg_df[['bill_length_mm','bill_depth_mm','flipper_length_mm']]; ykr=knn_reg_df['body_mass_g']
cv=KFold(n_splits=10,shuffle=True,random_state=RANDOM_STATE)
rows=[]
for k in range(1,42,2):
    m=Pipeline([('scale',StandardScaler()),('knn',KNeighborsRegressor(n_neighbors=k))])
    rmse=np.sqrt(-cross_val_score(m,Xkr,ykr,cv=cv,scoring='neg_mean_squared_error'))
    rows.append({'K':k,'CV_RMSE':rmse.mean(),'CV_sd':rmse.std(ddof=1)})
knn_reg_results=pd.DataFrame(rows)
p=figure(width=830,height=420,title='KNN regression: K vs CV RMSE',x_axis_label='K',y_axis_label='CV RMSE')
p.line(knn_reg_results['K'],knn_reg_results['CV_RMSE'],line_width=3)
p.scatter(knn_reg_results['K'],knn_reg_results['CV_RMSE'],size=8)
show(p)

# 25. Curse of dimensionality - why KNN struggles as dimension increases

As dimension grows, neighbourhoods become sparse and the distinction between "near" and "far" weakens.

To make this visible, we add irrelevant Gaussian noise features to the penguin classifier and compare KNN with logistic regression.

In [38]:
base=class_df[class_features].reset_index(drop=True)
y_noise=class_df['species'].reset_index(drop=True)
rows=[]
for q in [0,2,5,10,20,40]:
    Xn=base.copy()
    for j in range(q):
        Xn[f'noise_{j}']=rng.normal(size=len(Xn))
    knn=Pipeline([('scale',StandardScaler()),('model',KNeighborsClassifier(n_neighbors=7))])
    log=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=5000))])
    for name,m in [('KNN',knn),('Logistic',log)]:
        score=cross_val_score(m,Xn,y_noise,cv=skf,scoring='f1_macro').mean()
        rows.append({'noise_dimensions':q,'model':name,'macro_F1':score})
curse=pd.DataFrame(rows)
p=figure(width=850,height=430,title='Adding irrelevant dimensions: KNN vs logistic regression',x_axis_label='Number of added noise dimensions',y_axis_label='CV macro-F1')
for name,col in [('KNN','red'),('Logistic','indigo')]:
    part=curse[curse['model']==name]
    p.line(part['noise_dimensions'],part['macro_F1'],line_width=3,legend_label=name,color=col)
    p.scatter(part['noise_dimensions'],part['macro_F1'],size=8)
p.legend.location='bottom_left'; show(p)

# 26. Principal Component Analysis (PCA)

PCA constructs orthogonal directions

$$
Z_m=\phi_{1m}X_1+\cdots+\phi_{pm}X_p
$$

that successively maximise predictor variance.

Important ideas:

- standardise when scales differ,
- the loading vector has unit norm,
- PC signs are arbitrary,
- explained variance ratio measures how much variation each PC captures,
- there is no universally correct number of PCs,
- scree plots and cumulative variance are heuristics, not proof.

In [39]:
pca_df=df[class_features+['species']].dropna().copy()
X_pca=pca_df[class_features]
scaler=StandardScaler(); X_scaled=scaler.fit_transform(X_pca)
pca=PCA().fit(X_scaled); scores=pca.transform(X_scaled)
pve=pca.explained_variance_ratio_
pve_df=pd.DataFrame({
    'component':[f'PC{i+1}' for i in range(len(pve))],
    'PVE':pve,
    'cumulative_PVE':np.cumsum(pve)
})
show_table(pve_df,'PCA explained variance')
loadings=pd.DataFrame(pca.components_.T,index=class_features,columns=[f'PC{i+1}' for i in range(len(class_features))]).reset_index(names='feature')
show_table(loadings,'PCA loadings')

In [40]:
components=pve_df['component'].tolist()
p=figure(x_range=components,width=820,height=420,title='Scree / cumulative explained variance',x_axis_label='Principal component',y_axis_label='Variance proportion')
p.vbar(x=components,top=pve_df['PVE'],width=0.65,legend_label='individual PVE')
p.line(components,pve_df['cumulative_PVE'],line_width=3,legend_label='cumulative PVE')
p.scatter(components,pve_df['cumulative_PVE'],size=8)
p.legend.location='center_right'; show(p)

score_df=pd.DataFrame({'PC1':scores[:,0],'PC2':scores[:,1],'species':pca_df['species'].values})
p=figure(width=850,height=500,title='PCA projection - species shown only after fitting',x_axis_label='PC1',y_axis_label='PC2')
for s in sorted(score_df['species'].unique()):
    part=score_df[score_df['species']==s]
    p.scatter('PC1','PC2',source=ColumnDataSource(part),size=8,alpha=0.7,color=species_color[s],legend_label=s)
p.legend.location='top_left'; show(p)

## 26.1 Why scaling matters for PCA

Without scaling, variables measured in large numerical units can dominate the variance objective.

We compare PCA on raw measurements with PCA on standardized measurements.

In [41]:
pca_raw=PCA().fit(X_pca)
pve_compare=pd.DataFrame({
    'component':[f'PC{i+1}' for i in range(len(class_features))],
    'raw_PVE':pca_raw.explained_variance_ratio_,
    'standardized_PVE':pve,
})
show_table(pve_compare,'Raw-scale vs standardized PCA')

# 27. K-means clustering

K-means minimises within-cluster squared Euclidean variation:

$$
\sum_{k=1}^{K}\sum_{i\in C_k}\|x_i-\mu_k\|^2.
$$

Algorithm:

1. initialise centroids,
2. assign each point to its nearest centroid,
3. recompute centroids,
4. repeat until stable.

Important special cases:

- different initialisations can converge to different local minima,
- cluster labels are arbitrary,
- scaling changes the geometry,
- inertia always decreases with $K$, so minimum inertia alone cannot choose $K$.

In [42]:
cluster_rows=[]
for k in range(2,8):
    km=KMeans(n_clusters=k,n_init=30,random_state=RANDOM_STATE)
    lab=km.fit_predict(X_scaled)
    cluster_rows.append({'K':k,'inertia':km.inertia_,'silhouette':silhouette_score(X_scaled,lab)})
cluster_eval=pd.DataFrame(cluster_rows)
show_table(cluster_eval,'Choosing K: inertia and silhouette')

p1=figure(width=760,height=370,title='K-means inertia',x_axis_label='K',y_axis_label='Inertia')
p1.line(cluster_eval['K'],cluster_eval['inertia'],line_width=3); p1.scatter(cluster_eval['K'],cluster_eval['inertia'],size=8)
p2=figure(width=760,height=370,title='Silhouette score',x_axis_label='K',y_axis_label='Silhouette')
p2.line(cluster_eval['K'],cluster_eval['silhouette'],line_width=3); p2.scatter(cluster_eval['K'],cluster_eval['silhouette'],size=8)
show(column(p1,p2))

In [43]:
km3=KMeans(n_clusters=3,n_init=30,random_state=RANDOM_STATE)
cluster3=km3.fit_predict(X_scaled)
cluster_table=(pd.crosstab(pca_df['species'],cluster3,rownames=['species'],colnames=['cluster'])
               .rename(columns={0:'Cluster 0',1:'Cluster 1',2:'Cluster 2'}).reset_index())
show_table(cluster_table,'Clusters vs species - post-hoc interpretation only')

## 27.1 Local-minimum special case

`n_init=1` trusts one random start. `n_init=30` keeps the best of many starts.

Repeated single-start runs expose the optimisation instability directly.

In [44]:
inertias=[]
for seed in range(40):
    k=KMeans(n_clusters=3,n_init=1,random_state=seed).fit(X_scaled)
    inertias.append(k.inertia_)
best_multi=KMeans(n_clusters=3,n_init=30,random_state=RANDOM_STATE).fit(X_scaled).inertia_
show_table(pd.DataFrame({
    'statistic':['single-start min','single-start mean','single-start max','30-start selected inertia'],
    'inertia':[np.min(inertias),np.mean(inertias),np.max(inertias),best_multi]
}),'K-means initialisation sensitivity')

# 28. Hierarchical clustering and linkage choices

Agglomerative hierarchical clustering starts with every observation as its own cluster and repeatedly merges clusters.

Common linkage rules:

### Single linkage

$$
d(A,B)=\min_{a\in A,b\in B}d(a,b).
$$

### Complete linkage

$$
d(A,B)=\max_{a\in A,b\in B}d(a,b).
$$

### Average linkage

$$
d(A,B)=\frac{1}{|A||B|}\sum_{a\in A}\sum_{b\in B}d(a,b).
$$

### Ward linkage

Chooses the merge producing the smallest increase in within-cluster squared variation.

Different linkages encode different notions of cluster similarity and can produce visibly different dendrograms.

In [45]:
sample_idx=rng.choice(len(X_scaled),size=55,replace=False)
Xsamp=X_scaled[sample_idx]
figs=[]
for method in ['single','complete','average','ward']:
    Z=linkage(Xsamp,method=method)
    d=dendrogram(Z,no_plot=True)
    p=figure(width=430,height=330,title=f'{method.title()} linkage',x_axis_label='Sample order',y_axis_label='Merge distance')
    for xs,ys in zip(d['icoord'],d['dcoord']):
        p.line(xs,ys,line_width=1.7)
    p.xaxis.visible=False
    figs.append(p)
show(gridplot([figs[:2],figs[2:]]))

# 29. End-to-end concept map

| Concept | Core question | Main failure mode / trade-off |
|---|---|---|
| Simple regression | Is a linear trend useful? | misspecification |
| Regression inference | Is the estimated association precise? | invalid assumptions / unstable SEs |
| Multiple regression | What is the partial association? | collinearity / omitted interactions |
| Polynomial regression | Do we need more flexibility? | overfitting |
| CV | How well does the procedure generalise? | leakage / computational cost |
| Bootstrap | How uncertain is an estimator? | wrong resampling unit |
| Subset selection | Which predictors should remain? | combinatorial search / selection bias |
| Ridge | Can shrinkage reduce variance? | added bias |
| Lasso | Can we shrink and select? | unstable selection among correlated features |
| PCR | Can predictor variation be compressed? | PCs may not predict $Y$ |
| PLS | Can supervised latent directions help? | tuning / interpretation |
| Logistic regression | What are class probabilities / odds? | linear log-odds assumption |
| LDA | Can a shared-covariance Gaussian model classify well? | covariance assumption |
| QDA | Are class-specific covariances useful? | higher variance |
| KNN | Can local neighbourhoods predict? | curse of dimensionality |
| PCA | What latent variation dominates $X$? | scale and interpretability |
| K-means | Are there centroid-like groups? | local minima / $K$ choice |
| Hierarchical clustering | Is there nested group structure? | linkage sensitivity |

# 30. Exam-oriented special cases to remember

These are especially useful because they turn general ideas into memorable identities.

1. **Simple OLS line passes through the sample means**

$$
(\bar x,\bar y).
$$

2. **Simple regression with intercept**

$$
R^2=r_{XY}^2.
$$

3. **OLS LOOCV shortcut**

$$
e_{(i)}=\frac{e_i}{1-h_{ii}}.
$$

4. **Orthogonal-design Ridge**

$$
\hat\beta_j^{ridge}=\frac{\hat\beta_j^{OLS}}{1+\lambda}.
$$

5. **Orthogonal-design Lasso** is soft thresholding.

6. **Backward selection needs an estimable full model**, so it is unsuitable when $p\ge n$.

7. **PCA maximises variance of $X$, not predictive association with $Y$**.

8. **LDA vs QDA** is a bias-variance trade-off: shared covariance reduces parameters; class-specific covariance increases flexibility.

9. **KNN**: small $K$ = low bias/high variance; large $K$ = higher bias/lower variance.

10. **Cluster labels are arbitrary**; cluster 0 has no intrinsic meaning.

# 31. Student exercises

## Exercise A - regression inference

Re-fit the body-mass model with and without `island`. Compare:

- coefficient estimates,
- standard errors,
- adjusted $R^2$,
- overall $F$ statistic.

Explain why a variable can improve prediction while complicating scientific interpretation.

## Exercise B - heteroscedasticity

Create a synthetic regression dataset in which

$$
Var(\epsilon\mid X=x)
$$

increases with $x$. Compare ordinary standard errors with heteroscedasticity-robust standard errors.

## Exercise C - stepwise selection instability

Bootstrap the data 100 times and run forward AIC selection on each bootstrap sample. Count how frequently each predictor is selected.

## Exercise D - nested CV

Use inner CV to choose Ridge/Lasso strength and outer CV to estimate the entire tuning procedure.

## Exercise E - LDA/QDA sample-size trade-off

Subsample the penguin data to smaller training sets. Determine whether QDA deteriorates faster than LDA.

## Exercise F - KNN and dimensions

Repeat the curse-of-dimensionality experiment using 100 irrelevant features.

## Exercise G - PCR vs PLS

Explain why PLS can outperform one-component PCR in the synthetic low-variance-signal example.

## Exercise H - clustering stability

Repeat K-means with bootstrap-resampled observations and compare how stable the cluster assignments are.

# 32. Closing perspective

The cheatsheet becomes much easier to understand when its formulas are organised around three recurring questions.

### 1. How flexible should the learner be?

$$
\text{bias}\quad\leftrightarrow\quad\text{variance}
$$

### 2. How do we estimate performance honestly?

$$
\text{validation}\rightarrow\text{cross-validation}\rightarrow\text{test set}
$$

### 3. What structure are we assuming?

- linear mean function,
- Gaussian class-conditional distributions,
- local neighbourhood smoothness,
- low-dimensional latent structure,
- centroid-based or hierarchical clusters.

A strong ST3248 student should be able to move from a formula to:

- its intuition,
- its assumptions,
- its implementation,
- its diagnostics,
- its failure modes,
- and the appropriate resampling strategy.

# References

- Uploaded **ST3248 Statistical Learning cheatsheet** (Studocu; third-party study material).
- James, Witten, Hastie, Tibshirani and Taylor, *An Introduction to Statistical Learning*.
- Palmer Penguins dataset (`palmerpenguins`).
- scikit-learn documentation.
- statsmodels documentation.
- Bokeh documentation.

Dataset source used by the notebook:

`https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv`